In [ ]:
!pip install gensim
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 109.1 MB/s eta 0:00:00


In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng') # Bản cập nhật cho tiếng Anh

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
from __future__ import absolute_import, division, print_function
import codecs
import glob
import logging
import multiprocessing
import os
import pprint
import re
import keras
import nltk
import spacy
from nltk import ne_chunk, pos_tag
from nltk.tokenize import sent_tokenize, word_tokenize, PunktSentenceTokenizer
from nltk.corpus import stopwords
import gensim.models.word2vec as w2v
from gensim import logging
import sklearn.manifold
import numpy as np
import matplotlib.pyplot as plt
import pandas as  pd
import seaborn as sns
import pickle
import json
from nltk.tokenize import WhitespaceTokenizer
from nltk.tokenize import MWETokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = '/content/drive/MyDrive/Thesis/Agoda_Reviews_cleaned_v3.xlsx'
df_reviews = pd.read_excel(file_path)
reviews_list = df_reviews['clean_comments'].astype(str).tolist()

In [ ]:
def load_keywords(file_name):
    try:
        with open(file_name, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        return [x.strip() for x in lines if x.strip()]
    except FileNotFoundError:
        print(f"Không tìm thấy file {file_name}")
        return []

word_list_name = load_keywords('/content/drive/MyDrive/Thesis/Vocab/hotel_names_total.txt')
word_list_fac = load_keywords('/content/drive/MyDrive/Thesis/Vocab/facilities.txt')
word_list_loc = load_keywords('/content/drive/MyDrive/Thesis/Vocab/locations.txt')
word_list_price = load_keywords('/content/drive/MyDrive/Thesis/Vocab/Price.txt')
word_list_rel = load_keywords('/content/drive/MyDrive/Thesis/Vocab/located_relation.txt')
word_list_fac_rel = load_keywords('/content/drive/MyDrive/Thesis/Vocab/facility_relation.txt')
word_list_target = load_keywords('/content/drive/MyDrive/Thesis/Vocab/suitable_for.txt')

# lọc trùng và giữ nguyên thứ tự
def ordered_set(in_list):
    out_list = []
    added = set()
    for val in in_list:
        if not val in added:
            out_list.append(val)
            added.add(val)
    return out_list

all_keywords = word_list_name + word_list_fac + word_list_loc +word_list_rel + word_list_fac_rel + word_list_target
mw_list = []

for x in all_keywords:
    ws = x.split()
    if len(ws) > 1:
        # Chuyển thành tuple để spaCy hiểu đây là cụm từ cố định
        mw_list.append(tuple(ws))

mw_list = ordered_set(mw_list)

print(f"Đã chuẩn bị xong mw_list với {len(mw_list)} cụm từ.")
print("Ví dụ 5 cụm đầu tiên:", mw_list[0:10])



Đã chuẩn bị xong mw_list với 18926 cụm từ.
Ví dụ 5 cụm đầu tiên: [('118', 'Orchid'), ('1994', 'House'), ('2', 'Beautiful', 'Beach', 'View', 'bedrooms', 'New', 'Apartment'), ('20', 'Hotel', 'and', 'Apartment'), ('2001', 'Motel'), ('2005', 'Sea', '&', 'Horizon', 'Luxury', 'Apt', 'Full', 'Option'), ('22', 'Land', '71', 'Hang', 'Bong', 'Hotel'), ('22Land', 'Residence', 'Hotel'), ('22Housing', '39', 'Linh', 'Lang', 'Residence'), ('22Land', 'Classic', 'Suites')]


In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize, MWETokenizer
from tqdm import tqdm

# 1. Tải các tài nguyên cần thiết
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

name_set = set(w.lower().strip() for w in word_list_name)
loc_set = set(w.lower().strip() for w in word_list_loc)
fac_set = set(w.lower().strip() for w in word_list_fac)
target_set = set(w.lower().strip() for w in word_list_target)
price_set = set(w.lower().strip() for w in word_list_price)


# 3. Khởi tạo MWETokenizer để không bị tách các cụm từ (ví dụ: "Cam Ranh", "Movenpick Resort")
def prepare_mwe(word_lists):
    combined_list = []
    for wl in word_lists:
        combined_list.extend(wl)
    # Chỉ lấy các cụm từ có từ 2 từ trở lên
    return [tuple(word.lower().split()) for word in combined_list if len(word.split()) > 1]

mwe = MWETokenizer()
all_lists = [word_list_name, word_list_loc, word_list_fac, word_list_target]
for phrase in prepare_mwe(all_lists):
    mwe.add_mwe(phrase)

all_item = []

# 4. TIẾN HÀNH DÁN NHÃN TRÊN DỮ LIỆU REVIEWS
for i in tqdm(range(len(df_reviews))):
    # Lấy câu review và đưa về chữ thường để so sánh
    text = str(df_reviews.clean_comments[i]).lower()

    # Tokenize
    tokens_raw = word_tokenize(text)
    tokens_mwe = mwe.tokenize(tokens_raw)

    sentence_words = []
    sentence_tags = []

    for token in tokens_mwe:
        # MWETokenizer nối cụm bằng '_', ta chuyển lại thành khoảng trắng để check dictionary
        clean_token = token.replace('_', ' ').strip()
        subwords = clean_token.split()

        # Xác định nhãn gốc bằng cách so khớp với bộ Set đã chuẩn bị
        label = "O"
        if clean_token in name_set: label = "ORG"
        elif clean_token in loc_set: label = "LOC"
        elif clean_token in fac_set: label = "FACILITY"
        elif clean_token in target_set: label = "TARGET"
        elif clean_token in price_set: label = "PRICE"

        # Gán nhãn B- và I- cho từng subword
        for idx, sw in enumerate(subwords):
            sentence_words.append(sw)
            if label != "O":
                tag = f"B-{label}" if idx == 0 else f"I-{label}"
            else:
                tag = "O"
            sentence_tags.append(tag)

    # Lấy POS Tag cho các từ trong câu
    pos_tags = [p[1] for p in nltk.pos_tag(sentence_words)]

    # Lưu vào danh sách tổng
    for w, p, t in zip(sentence_words, pos_tags, sentence_tags):
        all_item.append({
            'Sentence #': i + 1,
            'Word': w,
            'POS': p,
            'Tag': t
        })

# 5. XUẤT FILE VÀ KIỂM TRA LẠI
df_output = pd.DataFrame(all_item)

print("\n--- THỐNG KÊ NHÃN SAU KHI CHẠY LẠI ---")
print(df_output['Tag'].value_counts())

# Kiểm tra xem có nhãn LOC và FACILITY chưa
unique_tags = df_output['Tag'].unique()
print(f"\nCác nhãn thực tế thu được: {unique_tags}")

# Lưu file chuẩn bị cho Training
df_output.to_csv('data-loc-fac-bert_FULL.csv', sep='*', index=False, encoding="utf-8")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
100%|██████████| 39205/39205 [00:52<00:00, 747.21it/s]



--- THỐNG KÊ NHÃN SAU KHI CHẠY LẠI ---
Tag
O             3622446
B-FACILITY     173300
B-ORG          108247
B-LOC           27733
I-FACILITY      27632
B-PRICE         23403
I-LOC           12578
B-TARGET        11544
I-ORG           11221
I-TARGET          484
Name: count, dtype: int64

Các nhãn thực tế thu được: ['B-ORG' 'O' 'I-ORG' 'B-FACILITY' 'I-FACILITY' 'B-PRICE' 'B-TARGET'
 'B-LOC' 'I-LOC' 'I-TARGET']


In [ ]:
print(f"Số lượng từ trong list Name: {len(word_list_name)}")
print(f"Số lượng từ trong list Loc: {len(word_list_loc)}")
print(f"Số lượng từ trong list Fac: {len(word_list_fac)}")
print(f"Số lượng từ trong list Target: {len(word_list_target)}")
print(f"Số lượng từ trong list price: {len(word_list_price)}")

Số lượng từ trong list Name: 10949
Số lượng từ trong list Loc: 8626
Số lượng từ trong list Fac: 496
Số lượng từ trong list Target: 81
Số lượng từ trong list price: 64


In [ ]:
import pandas as pd
import pandas as pd
import numpy as np
from tqdm import tqdm, trange
#load data
# explore data
file_csv = 'data-loc-fac-bert_FULL.csv'
data = pd.read_csv(file_csv, sep='*',encoding="utf-8").fillna(method="ffill")

data.head(1000)


/tmp/ipykernel_872/2170588429.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = pd.read_csv(file_csv, sep='*',encoding="utf-8").fillna(method="ffill")


,Sentence #,Word,POS,Tag
0,1,my,PRP$,B-ORG
1,1,wife,NN,O
2,1,and,CC,O
3,1,i,NN,O
4,1,recently,RB,O
...,...,...,...,...
995,3,for,IN,O
996,3,families,NNS,B-TARGET
997,3,and,CC,O
998,3,muslim,JJ,O


In [ ]:
import pandas as pd
from datasets import Dataset, Features, Sequence, Value, ClassLabel

# 1. Load dữ liệu
df = pd.read_csv('data-loc-fac-bert_FULL.csv', sep='*', encoding="utf-8")

# 2. Định nghĩa danh sách nhãn
label_list = ['O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-FACILITY', 'I-FACILITY', 'B-TARGET', 'I-TARGET','B-PRICE']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# 3. Gom nhóm dữ liệu
# Đảm bảo Word luôn là string và Tag luôn là id (int)
agg_func = lambda s: [
    [str(w) for w in s["Word"].values.tolist()],
    [label2id[t] for t in s["Tag"].values.tolist()]
]

grouped = df.groupby("Sentence #").apply(agg_func, include_groups=False)
sentences = [s[0] for s in grouped]
labels = [s[1] for s in grouped]

# 4. ĐỊNH NGHĨA FEATURES (Bước quan trọng nhất để sửa lỗi ArrowInvalid)
# Việc khai báo Features giúp thư viện datasets biết chính xác kiểu dữ liệu, không tự đoán nữa.
features = Features({
    "tokens": Sequence(Value("string")),
    "ner_tags": Sequence(ClassLabel(names=label_list))
})

raw_data = {"tokens": sentences, "ner_tags": labels}
full_ds = Dataset.from_dict(raw_data, features=features)

ds_split = full_ds.train_test_split(test_size=0.3, seed=42)

print(ds_split["train"][0])

{'tokens': ['the', 'villa', 'is', 'located', 'in', 'a', 'villa', 'area', ',', 'with', 'spacious', 'roads', 'and', 'comfortable', 'parking', 'for', 'cars', '.', 'its', 'conveniently', 'located', 'near', 'the', 'center', '.', 'the', 'villa', 'is', 'quite', 'lovely', ',', 'and', 'the', 'cafe', 'and', 'breakfast', 'area', 'has', 'a', 'nice', 'chill', 'view', '.', 'the', 'owner', 'takes', 'great', 'care', 'of', 'the', 'garden', 'and', 'the', 'furnishings', 'in', 'the', 'room', '.', 'the', 'breakfast', 'includes', 'dishes', 'like', 'pho', ',', 'beef', 'noodle', 'soup', ',', 'and', 'h', 't', 'u', ',', 'which', 'are', 'quite', 'delicious', '.', 'the', 'accommodation', 'is', 'suitable', 'for', 'guests', 'who', 'appreciate', 'tranquility', ',', 'as', 'there', 'are', 'no', 'restaurants', 'within', 'a', '1', 'kilometers', 'radius', 'in', 'the', 'evening', '.', 'with', 'the', 'room', 'price', 'including', 'breakfast', ',', 'it', 'is', 'worth', 'the', 'money', '.'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0,

In [ ]:
from transformers import AutoTokenizer

# bert-base-cased
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Ánh xạ token về từ gốc
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Token đặc biệt (CLS, SEP) gán nhãn -100 để bỏ qua khi tính loss
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx]) # Token đầu tiên của từ
            else:
                label_ids.append(label[word_idx]) # Các token phụ sau đó gán cùng nhãn (hoặc dùng nhãn I-)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Map toàn bộ dataset qua hàm tokenize
tokenized_datasets = ds_split.map(tokenize_and_align_labels, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/27443 [00:00<?, ? examples/s]

Map:   0%|          | 0/11762 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [ ]:
args = TrainingArguments(
    f"{model_checkpoint}-finetuned-ner",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    push_to_hub=False,
)

In [ ]:
!pip install evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=fd675f87935db482d29a266635c8298ec757bd072b8017d5d4d424b5cd15eb65
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import evaluate
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Loại bỏ các vị trí đặc biệt (-100) trước khi tính điểm
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
from transformers import DataCollatorForTokenClassification

# 1. Khai báo lại data_collator (vì đổi Runtime nên biến bị mất)
data_collator = DataCollatorForTokenClassification(tokenizer)

# 2. Khởi tạo lại Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# 3. Bắt đầu huấn luyện
trainer.train()




Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.022713,0.017337,0.966472,0.972623,0.969537,0.994555
2,0.011013,0.011967,0.977289,0.981917,0.979597,0.996286
3,0.006522,0.011006,0.982204,0.984882,0.983541,0.996869
4,0.004537,0.011544,0.980895,0.987050,0.983963,0.996961
5,0.002705,0.011940,0.982784,0.986803,0.984789,0.997077


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8580, training_loss=0.01569095135672943, metrics={'train_runtime': 856.5083, 'train_samples_per_second': 160.203, 'train_steps_per_second': 10.017, 'total_flos': 2.485830044225994e+16, 'train_loss': 0.01569095135672943, 'epoch': 5.0})

In [ ]:
save_directory = "/content/drive/MyDrive/Thesis/NER_Bert_Model_v3"

trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"lưu model tại: {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

lưu model tại: /content/drive/MyDrive/Thesis/NER_Bert_Model_v3


In [ ]:
from transformers import pipeline

# Gọi mô hình thẳng từ folder Drive
save_directory =  "/content/drive/MyDrive/Thesis/NER_Bert_Model_v3"
nlp_bert = pipeline("ner", model=save_directory, aggregation_strategy="simple")

# Chạy thử 1 câu
result = nlp_bert("As a solo traveler , my stay at Maris Hotel was genuinely excellent . From the moment I arrived")
print(result)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity_group': 'TARGET', 'score': np.float32(0.99709946), 'word': 'solo traveler', 'start': 5, 'end': 18}, {'entity_group': 'ORG', 'score': np.float32(0.9998468), 'word': 'my stay', 'start': 21, 'end': 28}, {'entity_group': 'ORG', 'score': np.float32(0.99968255), 'word': 'at', 'start': 29, 'end': 31}, {'entity_group': 'ORG', 'score': np.float32(0.8838374), 'word': 'Hotel', 'start': 38, 'end': 43}]


In [ ]:
import numpy as np
import pandas as pd
from transformers import AutoModelForTokenClassification, AutoTokenizer
from torch.utils.data import DataLoader
# Add necessary imports for Dataset operations
from datasets import Dataset, Features, Sequence, Value, ClassLabel

#!pip install seqeval # Already satisfied from previous execution
from seqeval.metrics import classification_report
import torch

# --- Start: Re-defining missing variables due to potential kernel state loss ---
# These definitions are copied from earlier cells (5_TEgyXqRL7u and xgefFS8BRPVG)
# to ensure 'tokenized_datasets' and 'label_list' are available in this scope.

# Re-define label_list and related data processing from cell 5_TEgyXqRL7u
df = pd.read_csv('data-loc-fac-bert_FULL.csv', sep='*', encoding="utf-8")
label_list = ['O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-FACILITY', 'I-FACILITY', 'B-TARGET', 'I-TARGET']
label2id = {label: i for i, label in enumerate(label_list)}

agg_func = lambda s: [
    [str(w) for w in s["Word"].values.tolist()],
    [label2id[t] for t in s["Tag"].values.tolist()]
]
grouped = df.groupby("Sentence #").apply(agg_func, include_groups=False)
sentences = [s[0] for s in grouped]
labels = [s[1] for s in grouped]

features = Features({
    "tokens": Sequence(Value("string")),
    "ner_tags": Sequence(ClassLabel(names=label_list))
})
raw_data = {"tokens": sentences, "ner_tags": labels}
full_ds = Dataset.from_dict(raw_data, features=features)
ds_split = full_ds.train_test_split(test_size=0.3, seed=42)

# Re-define tokenizer and tokenized_datasets from cell xgefFS8BRPVG
model_checkpoint = "bert-base-cased"
# Use a separate tokenizer for dataset preparation to avoid conflict with model-specific tokenizer
data_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = data_tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = ds_split.map(tokenize_and_align_labels, batched=True)
# --- End: Re-defining missing variables ---

# 1. Đường dẫn lưu model (original code starts here, with necessary adjustments)
save_directory = "/content/drive/MyDrive/Thesis/NER_Bert_Model"

# 2. Nạp nhanh Model và Tokenizer
model = AutoModelForTokenClassification.from_pretrained(save_directory)
# The tokenizer variable here is for the loaded model, different from data_tokenizer
tokenizer = AutoTokenizer.from_pretrained(save_directory)
model.eval() # Chuyển sang chế độ đánh giá
if torch.cuda.is_available():
    model.cuda()

# 3. Hàm dự đoán cho tập Test (Dùng tokenized_datasets['test'] đã có từ bước trước)
def get_predictions(dataset):
    all_predictions = []
    all_labels = []

    # Chuyển dataset sang dạng nôm na để duyệt (hoặc dùng DataLoader để nhanh hơn)
    for examples in dataset:
        input_ids = torch.tensor([examples["input_ids"]]).to(model.device)
        attention_mask = torch.tensor([examples["attention_mask"]]).to(model.device)
        labels = examples["labels"]

        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1).cpu().numpy()[0]

        # Chỉ lấy những vị trí không phải -100 (bỏ CLS, SEP, Padding)
        true_pred = [label_list[p] for (p, l) in zip(predictions, labels) if l != -100]
        true_lab = [label_list[l] for (p, l) in zip(predictions, labels) if l != -100]

        all_predictions.append(true_pred)
        all_labels.append(true_lab)

    return all_labels, all_predictions

# 4. Chạy đánh giá
print("--- Đang thực hiện đánh giá trên bộ Test ---")
y_true, y_pred = get_predictions(tokenized_datasets["test"])

# 5. Xuất báo cáo chi tiết theo nhãn (Chuẩn NCKH)
results = classification_report(y_true, y_pred, mode='strict', scheme='BIO', output_dict=True)

# Chuyển thành DataFrame để dán vào luận văn
report_df = pd.DataFrame(results).transpose()

print("\n--- BẢNG KẾT QUẢ CHI TIẾT (REPORT BY LABEL) ---\n")
display(report_df)

# Lưu lại kết quả để đối chiếu
report_df.to_csv(f"{save_directory}/evaluation_report_detailed.csv")

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification
from torch.utils.data import DataLoader
from datasets import Dataset, Features, Sequence, Value, ClassLabel
from seqeval.metrics import classification_report
import os

# --- 1. CẤU HÌNH ĐƯỜNG DẪN & NHÃN ---
save_directory = "/content/drive/MyDrive/Thesis/NER_Bert_Model"
data_path = 'data-loc-fac-bert_FULL.csv'
model_checkpoint = "bert-base-cased"

# Danh sách nhãn chuẩn (Phải khớp với lúc train)
label_list = ['O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-FACILITY', 'I-FACILITY', 'B-TARGET', 'I-TARGET']
label2id = {label: i for i, label in enumerate(label_list)}

# --- 2. TIỀN XỬ LÝ DỮ LIỆU TẬP TEST ---
print("--- Đang chuẩn bị dữ liệu Test ---")
df = pd.read_csv(data_path, sep='*', encoding="utf-8")

# Group dữ liệu theo câu
agg_func = lambda s: [
    [str(w) for w in s["Word"].values.tolist()],
    [label2id[t] for t in s["Tag"].values.tolist()]
]
grouped = df.groupby("Sentence #").apply(agg_func, include_groups=False)
sentences = [s[0] for s in grouped]
labels = [s[1] for s in grouped]

# Tạo Dataset và chia tách (đảm bảo lấy đúng tập test 30% như lúc train)
features = Features({
    "tokens": Sequence(Value("string")),
    "ner_tags": Sequence(ClassLabel(names=label_list))
})
full_ds = Dataset.from_dict({"tokens": sentences, "ner_tags": labels}, features=features)
ds_split = full_ds.train_test_split(test_size=0.3, seed=42)

# --- 3. TOKENIZE & ALIGN LABELS ---
tokenizer = AutoTokenizer.from_pretrained(save_directory)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_test = ds_split["test"].map(tokenize_and_align_labels, batched=True)
# Remove columns not needed for model input to avoid issues with DataCollator
tokenized_test = tokenized_test.remove_columns(["tokens", "ner_tags"])

# --- 4. LOAD MODEL & SETUP DATALOADER (TỐI ƯU TỐC ĐỘ) ---
print("--- Đang nạp Model và chạy dự đoán siêu tốc ---")
model = AutoModelForTokenClassification.from_pretrained(save_directory)
if torch.cuda.is_available():
    model.cuda()

data_collator = DataCollatorForTokenClassification(tokenizer)
test_loader = DataLoader(tokenized_test, batch_size=32, collate_fn=data_collator)

# --- 5. HÀM EVALUEATE CHÍNH ---
def evaluate_fast(model, loader):
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            inputs = {k: v.to(model.device) for k, v in batch.items() if k != "labels"}
            labels = batch["labels"].to(model.device)

            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            labels = labels.cpu().numpy()

            for i in range(len(predictions)):
                # Lọc bỏ nhãn -100 để tính Precision/Recall chính xác
                true_pred = [label_list[p] for (p, l) in zip(predictions[i], labels[i]) if l != -100]
                true_lab = [label_list[l] for (p, l) in zip(predictions[i], labels[i]) if l != -100]
                all_predictions.append(true_pred)
                all_labels.append(true_lab)
    return all_labels, all_predictions

# Thực thi
y_true, y_pred = evaluate_fast(model, test_loader)

# --- 6. XUẤT BÁO CÁO CHI TIẾT ---
results = classification_report(y_true, y_pred, mode='strict', scheme='BIO', output_dict=True)
report_df = pd.DataFrame(results).transpose()

print("\n" + "="*50)
print("KẾT QUẢ ĐÁNH GIÁ CHI TIẾT THEO NHÃN (BERT)")
print("="*50)
display(report_df)

# Lưu kết quả vào Drive
output_file = os.path.join(save_directory, "evaluation_report_fast.csv")
report_df.to_csv(output_file)
print(f"\n✅ Đã lưu báo cáo tại: {output_file}")

####Evaluate theo từng loại nhãn

In [ ]:
from seqeval.metrics import classification_report
import numpy as np
import time

start_time = time.time()

# 1. Chạy dự đoán trực tiếp từ Trainer trên tập Test
print("Đang chạy dự đoán trên tập Test...")
outputs = trainer.predict(tokenized_datasets["test"])

logits = outputs.predictions[0] if isinstance(outputs.predictions, tuple) else outputs.predictions
predictions = np.argmax(logits, axis=-1)
labels = outputs.label_ids

# 2. Khôi phục lại nhãn dạng chữ và LOẠI BỎ các token bị phớt lờ (-100)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

# 3. In bảng báo cáo chi tiết TỪNG NHÃN
print("\n" + "="*50)
print("BẢNG ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN CHO BERT (Dùng seqeval)")
print("="*50)
print(classification_report(true_labels, true_predictions, zero_division=0))

print(f"\n⏱️ Thời gian xử lý: {time.time() - start_time:.4f} giây")

Đang chạy dự đoán trên tập Test...



BẢNG ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN CHO BERT (Dùng seqeval)
              precision    recall  f1-score   support

    FACILITY       0.99      1.00      0.99     55946
         LOC       0.96      0.97      0.96     10735
         ORG       0.97      0.98      0.97     35590
       PRICE       1.00      1.00      1.00      7217
      TARGET       0.99      1.00      0.99      3488

   micro avg       0.98      0.99      0.98    112976
   macro avg       0.98      0.99      0.98    112976
weighted avg       0.98      0.99      0.98    112976


⏱️ Thời gian xử lý: 34.6084 giây


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# 1. Dự đoán trực tiếp trên tập Test đã chuẩn hóa (Siêu nhanh)
print(f"Đang đánh giá trực tiếp từ kiến trúc model...")
outputs = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(outputs.predictions, axis=-1)
labels = outputs.label_ids

# 2. Khử token đặc biệt (-100)
mask = labels != -100
y_true = labels[mask]
y_pred = predictions[mask]

# 3. Lấy đúng danh sách tên nhãn từ model config
# Điều này đảm bảo ID 1 là B-LOC thì dự đoán 1 cũng hiện là B-LOC
target_names = [model.config.id2label[i] for i in range(len(model.config.id2label))]

# 4. In báo cáo (Loại bỏ nhãn 'O' để soi kỹ các thực thể)
labels_ids_to_show = [i for i, label in model.config.id2label.items() if label != 'O']
target_names_to_show = [model.config.id2label[i] for i in labels_ids_to_show]

print("\n--- BÁO CÁO KẾT QUẢ CHUẨN ĐẾN TỪNG TOKEN ---")
print(classification_report(y_true, y_pred,
                            labels=labels_ids_to_show,
                            target_names=target_names_to_show,
                            zero_division=0))

Đang đánh giá trực tiếp từ kiến trúc model...



--- BÁO CÁO KẾT QUẢ CHUẨN ĐẾN TỪNG TOKEN ---
              precision    recall  f1-score   support

       B-LOC       0.97      0.97      0.97     10735
       I-LOC       0.93      0.95      0.94      4831
       B-ORG       0.98      0.98      0.98     35590
       I-ORG       0.86      0.87      0.86      4999
  B-FACILITY       1.00      1.00      1.00     55946
  I-FACILITY       0.98      0.99      0.99      8789
    B-TARGET       0.99      1.00      0.99      3488
    I-TARGET       0.97      0.99      0.98       190
     B-PRICE       1.00      1.00      1.00      7217

   micro avg       0.98      0.98      0.98    131785
   macro avg       0.96      0.97      0.97    131785
weighted avg       0.98      0.98      0.98    131785

